In [1]:
import numpy as np 
import pandas as pd 
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose, DecomposeResult
from darts.timeseries import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel

/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('electricity.csv', index_col = 'ds', parse_dates = True)
df = df.query('unique_id == "DE"')

Exploratory Data Analysis

In [3]:
fig = px.line(df['y'], title="Hourly Electricity Price")
fig.update_layout(yaxis_title="", showlegend=False)
fig.show()

In [4]:
def plotly_plot(self):
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True)

    fig.add_trace(px.line(self.observed).data[0],row=1, col=1)
    fig.update_yaxes(title_text='Observed', row=1, col=1)

    fig.add_trace(px.line(self.trend).data[0], row=2, col=1)
    fig.update_yaxes(title_text='Trend', row=2, col=1)

    fig.add_trace(px.line(self.seasonal).data[0], row=3, col=1)
    fig.update_yaxes(title_text='Seasonal', row=3, col=1)

    fig.add_trace(px.scatter(self.resid).data[0], row=4, col=1)
    fig.update_yaxes(title_text='Resid', row=4, col=1)

    fig.update_layout(title_text='y', title_x=0.5, showlegend=False, height=750)

    fig.show()

setattr(DecomposeResult, 'plotly_plot', plotly_plot)

In [5]:
decomposition = seasonal_decompose(df['y'], model='add', period=24)
decomposition.plotly_plot()

In [6]:
def plotly_acf(series, lags=40):
    acf_values = acf(series, nlags=lags)
    lags_values = np.arange(0, lags + 1)

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=lags_values,
        y=acf_values,
        name='',
        marker=dict(color='black'),
        width=0.3
    ))

    fig.add_trace(go.Scatter(
        x=lags_values,
        y=acf_values,
        mode='markers',
        name='',
        marker=dict(color='blue', size=5)
    ))

    fig.update_layout(
        showlegend=False,
        yaxis=dict(range=[-1, 1])
    )
    
    return fig

def plotly_pacf(series, lags=40):
    pacf_values = pacf(series, nlags=lags)
    lags_values = np.arange(0, lags + 1)

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=lags_values,
        y=pacf_values,
        name='',
        marker=dict(color='black'),
        width=0.3
    ))

    fig.add_trace(go.Scatter(
        x=lags_values,
        y=pacf_values,
        mode='markers',
        name='',
        marker=dict(color='blue', size=5)
    ))

    fig.update_layout(
        showlegend=False,
        yaxis=dict(range=[-1, 1]))
    
    return fig

In [7]:
fig = make_subplots(rows=2, cols=1)

fig.add_trace(plotly_acf(df['y'], lags=100).data[0], row=1, col=1)
fig.add_trace(plotly_acf(df['y'], lags=100).data[1], row=1, col=1)

fig.add_trace(plotly_pacf(df['y'], lags=100).data[0], row=2, col=1)
fig.add_trace(plotly_pacf(df['y'], lags=100).data[1], row=2, col=1)

fig.update_layout(
    showlegend=False,
    yaxis=dict(range=[-1, 1]),
    height=750)

fig.update_yaxes(range=[-1, 1], row=1, col=1)
fig.update_yaxes(range=[-1, 1], row=2, col=1)

Series and Seasonality

In [8]:
series = TimeSeries.from_dataframe(df,
                                   value_cols = 'y',
                                   freq = 'h')

def encode_year(idx):
    return (idx.year - 2000) / 50

add_encoders = {
    'cyclic': {'future': ['hour', 'day', 'dayofweek', 'week', 'month']},
    'datetime_attribute': {'future': ['hour', 'day', 'dayofweek', 'week', 'month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {'past': [encode_year], 'future': [encode_year]},
    'transformer': Scaler(),
    'tz': 'CET'
}

Past Covariates

In [9]:
X_past = df.iloc[:,2:]
past_covariates = TimeSeries.from_dataframe(X_past, 
                                            freq='h')

Scaling

In [10]:
scaler_y = Scaler()
scaler_covariates = Scaler()

y_transformed = scaler_y.fit_transform(series)
past_covariates_transformed = scaler_covariates.fit_transform(past_covariates)

In [11]:
forecast_horizon = 24

In [12]:
model = NBEATSModel(
    input_chunk_length = 96,
    output_chunk_length = forecast_horizon,
    add_encoders = add_encoders,
    random_state = 42,
    n_epochs = 10,
    batch_size = 64,
    num_stacks = 30,
    num_blocks = 1,
    num_layers = 4,
    layer_widths = 512,
    pl_trainer_kwargs = {'accelerator': 'cpu'}
)

model.fit(y_transformed,
          past_covariates = past_covariates_transformed)

Specified future encoders in `add_encoders` at model creation but model does not accept future covariates. future encoders will be ignored.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: PossibleUserWarning:

GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 31.3 M | train
-------------------------------------------------------------
31.3 M    

Epoch 9: 100%|██████████| 25/25 [00:04<00:00,  5.93it/s, train_loss=0.00451]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 25/25 [00:04<00:00,  5.93it/s, train_loss=0.00451]


NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=1, num_layers=4, layer_widths=512, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, input_chunk_length=96, output_chunk_length=24, add_encoders={'cyclic': {'future': ['hour', 'day', 'dayofweek', 'week', 'month']}, 'datetime_attribute': {'future': ['hour', 'day', 'dayofweek', 'week', 'month']}, 'position': {'past': ['relative'], 'future': ['relative']}, 'custom': {'past': [<function encode_year at 0x7f702acc5440>], 'future': [<function encode_year at 0x7f702acc5440>]}, 'transformer': Scaler, 'tz': 'CET'}, random_state=42, n_epochs=10, batch_size=64, pl_trainer_kwargs={'accelerator': 'cpu'})

Cross-Validation

In [16]:
cv = model.historical_forecasts(
    series = y_transformed,
    past_covariates = past_covariates_transformed,
    start = df.shape[0] - (10*forecast_horizon),
    forecast_horizon = forecast_horizon,
    stride = forecast_horizon,
    retrain = True,
    last_points_only = False
)

`enable_optimization=True` is ignored because `retrain` is not `False` or `0`. To hide this warning, set `show_warnings=False` or `enable_optimization=False`.
Specified future encoders in `add_encoders` at model creation but model does not accept future covariates. future encoders will be ignored.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: PossibleUserWarning:

GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | Met

Epoch 9: 100%|██████████| 21/21 [00:03<00:00,  6.11it/s, train_loss=0.00387]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 21/21 [00:03<00:00,  6.11it/s, train_loss=0.00387]

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


...

Epoch 9: 100%|██████████| 25/25 [00:04<00:00,  5.56it/s, train_loss=0.00156]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 25/25 [00:04<00:00,  5.56it/s, train_loss=0.00156]

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: PossibleUserWarning:

GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.



In [13]:
from sklearn.metrics import root_mean_squared_error

In [14]:
rmse_cv = []

for i in range(len(cv)):

    predictions = TimeSeries.pd_series(scaler_y.inverse_transform(cv[i]))

    start = predictions.index.min()
    end = predictions.index.max()
    actuals = df.loc[start:end, 'y']

    rmse = root_mean_squared_error(actuals, predictions)
    rmse_cv.append(rmse)

print(rmse_cv)
print(np.mean(rmse_cv))

NameError: name 'cv' is not defined

In [13]:
import matplotlib.pyplot as plt

In [15]:
import plotly.graph_objects as go

fig = go.Figure()

for i in range(len(cv)):
    predictions = TimeSeries.pd_series(scaler_y.inverse_transform(cv[i]))
    start = predictions.index.min()
    end = predictions.index.max()
    actuals = df.loc[start:end, 'y']
    
    fig.add_trace(go.Scatter(x=actuals.index, y=actuals.values, mode='lines', name=f'Actuals {i+1}', line_color='black'))
    fig.add_trace(go.Scatter(x=predictions.index, y=predictions.values, mode='lines', name=f'Predictions {i+1}'))

fig.update_layout(title='CV Results',
                  title_x = 0.5,
                  showlegend=False)

fig.show()


NameError: name 'cv' is not defined

Parameter Tuning

In [16]:
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import root_mean_squared_error

In [18]:
param_grid = {
    'input_chunk_length': [72, 96],
    'n_epochs': [10],
    'batch_size': [32, 64],
    'num_blocks': [2, 3],
    'num_layers': [2, 4, 6],
    'layer_widths': [256, 512, 1024]
}

In [27]:
fixed_params = {
    'output_chunk_length': forecast_horizon,
    'add_encoders': add_encoders,
    'random_state': 42,
    'pl_trainer_kwargs': {'accelerator': 'cpu'}
}

In [28]:
n_iter = 10

param_list = list(ParameterSampler(param_grid,
                                   n_iter=n_iter,
                                   random_state=42))

In [29]:
param_list1 = param_list[:5]
param_list2 = param_list[5:]

In [30]:
for params in param_list1:
    params.update(fixed_params)

for params in param_list2:
    params.update(fixed_params)

In [25]:
param_list2

[{'num_layers': 6,
  'num_blocks': 2,
  'n_epochs': 10,
  'layer_widths': 1024,
  'input_chunk_length': 72,
  'batch_size': 64,
  'output_chunk_length': 24,
  'add_encoders': {'cyclic': {'future': ['hour',
     'day',
     'dayofweek',
     'week',
     'month']},
   'datetime_attribute': {'future': ['hour',
     'day',
     'dayofweek',
     'week',
     'month']},
   'position': {'past': ['relative'], 'future': ['relative']},
   'custom': {'past': [<function __main__.encode_year(idx)>],
    'future': [<function __main__.encode_year(idx)>]},
   'transformer': Scaler,
   'tz': 'CET'},
  'random_state': 42,
  'pl_trainer_kwargs': {'accelerator': 'gpu'}},
 {'num_layers': 4,
  'num_blocks': 3,
  'n_epochs': 10,
  'layer_widths': 512,
  'input_chunk_length': 72,
  'batch_size': 32,
  'output_chunk_length': 24,
  'add_encoders': {'cyclic': {'future': ['hour',
     'day',
     'dayofweek',
     'week',
     'month']},
   'datetime_attribute': {'future': ['hour',
     'day',
     'dayofweek',

In [ ]:
total_rmse = []

for num, params in enumerate(param_list1):
    

    print(f'\n\n\n\n\n\nTuning params set {num}\n\n\n\n\n\n')
    
    model = NBEATSModel(**params)


    cv = model.historical_forecasts(series=y_transformed,
                                    past_covariates=past_covariates_transformed,
                                    start=df.shape[0] - (10 * forecast_horizon),
                                    forecast_horizon = forecast_horizon,
                                    stride = forecast_horizon,
                                    retrain = True,
                                    last_points_only = False)
    
    rmse_cv = []

    for i in range(len(cv)):
        
        predictions = TimeSeries.pd_series(scaler_y.inverse_transform(cv[i]))

        start = predictions.index.min()
        end = predictions.index.max()
        actuals = df.loc[start:end, 'y']

        rmse = root_mean_squared_error(actuals, predictions)
        rmse_cv.append(rmse)

    mean_rmse = np.mean(rmse_cv)
    total_rmse.append(mean_rmse)

`enable_optimization=True` is ignored because `retrain` is not `False` or `0`. To hide this warning, set `show_warnings=False` or `enable_optimization=False`.
Specified future encoders in `add_encoders` at model creation but model does not accept future covariates. future encoders will be ignored.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs








Tuning params set 0








/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: PossibleUserWarning:

GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 26.6 M | train
-------------------------------------------------------------
26.6 M    Trainable params
3.4 K     Non-trainable params
26.6 M    Total params
106.287   Total estimated model params size (MB)
1056      Modules in train mode
0         Modules in eval mode


Epoch 9: 100%|██████████| 43/43 [00:08<00:00,  5.31it/s, train_loss=0.00417]

`Trainer.fit` stopped: `max_epochs=10` reached.


...

Epoch 9: 100%|██████████| 49/49 [00:21<00:00,  2.24it/s, train_loss=0.00715]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 49/49 [00:21<00:00,  2.24it/s, train_loss=0.00715]

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: PossibleUserWarning:

GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.



In [39]:
results_df = pd.DataFrame(param_list1)
results_df['rmse'] = total_rmse

best_params = results_df.sort_values(by = 'rmse').head(1)
best_params

,num_layers,num_blocks,n_epochs,layer_widths,input_chunk_length,batch_size,output_chunk_length,add_encoders,random_state,pl_trainer_kwargs,rmse
3,2,2,10,256,72,32,24,"{'cyclic': {'future': ['hour', 'day', 'dayofwe...",42,{'accelerator': 'cpu'},22.154044


In [53]:
best_params_dict = best_params.iloc[:,:7].squeeze().to_dict()
best_params_dict

{'num_layers': 2,
 'num_blocks': 2,
 'n_epochs': 10,
 'layer_widths': 256,
 'input_chunk_length': 72,
 'batch_size': 32,
 'output_chunk_length': 24}

In [54]:
for key, value in best_params_dict.items():
    best_params_dict[key] = int(value)

best_params_dict.update(fixed_params)

In [55]:
tuned_model = NBEATSModel(**best_params_dict)

tuned_model.fit(y_transformed, past_covariates = past_covariates_transformed)

Specified future encoders in `add_encoders` at model creation but model does not accept future covariates. future encoders will be ignored.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: PossibleUserWarning:

GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 9.8 M  | train
-------------------------------------------------------------
9.8 M     

Epoch 9: 100%|██████████| 50/50 [00:04<00:00, 10.72it/s, train_loss=0.00761]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 50/50 [00:04<00:00, 10.72it/s, train_loss=0.00761]


NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=2, num_layers=2, layer_widths=256, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, n_epochs=10, input_chunk_length=72, batch_size=32, output_chunk_length=24, add_encoders={'cyclic': {'future': ['hour', 'day', 'dayofweek', 'week', 'month']}, 'datetime_attribute': {'future': ['hour', 'day', 'dayofweek', 'week', 'month']}, 'position': {'past': ['relative'], 'future': ['relative']}, 'custom': {'past': [<function encode_year at 0x7f702acc5440>], 'future': [<function encode_year at 0x7f702acc5440>]}, 'transformer': Scaler, 'tz': 'CET'}, random_state=42, pl_trainer_kwargs={'accelerator': 'cpu'})

In [56]:
# Forecast the future
forecast = tuned_model.predict(n = forecast_horizon,
                               series = y_transformed,
                               past_covariates = past_covariates_transformed)

forecast =TimeSeries.pd_series(scaler_y.inverse_transform(forecast)).rename('NBEATS')
forecast

GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/salgado/scripts/time_series/.venv/lib64/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: PossibleUserWarning:

GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 85.34it/s]


ds
2017-12-31 00:00:00   -12.857973
2017-12-31 01:00:00   -17.251101
2017-12-31 02:00:00    -8.399998
2017-12-31 03:00:00   -11.765345
2017-12-31 04:00:00    11.310185
2017-12-31 05:00:00   -24.591890
2017-12-31 06:00:00    -6.464322
2017-12-31 07:00:00   -16.114572
2017-12-31 08:00:00    -2.205924
2017-12-31 09:00:00    -6.795763
2017-12-31 10:00:00    -8.209700
2017-12-31 11:00:00   -23.427114
2017-12-31 12:00:00     3.116838
2017-12-31 13:00:00     3.101331
2017-12-31 14:00:00    22.462017
2017-12-31 15:00:00    11.100773
2017-12-31 16:00:00    -6.171403
2017-12-31 17:00:00    20.423751
2017-12-31 18:00:00    15.844217
2017-12-31 19:00:00    18.275454
2017-12-31 20:00:00    23.726588
2017-12-31 21:00:00     4.863163
2017-12-31 22:00:00     4.175991
2017-12-31 23:00:00    10.429275
Freq: h, Name: NBEATS, dtype: float64

In [85]:
fig = go.Figure()
fig.add_trace(px.line(df['y']['2017-12':]).data[0].update(line_color='black'))

fig.add_trace(px.line(forecast).data[0].update(line_dash='dot'))
fig.update_layout(showlegend=False,
                  title='Electricity Price Forecast',
                  title_x = 0.5)
